# Notebook #06 — Statistical Evaluation, Subgroup Audit & Model Governance Registry

### Nội dung cốt lõi
1. **Kiểm định thống kê độ tin cậy:** Paired Bootstrap 95% CI, McNemar Test, và Permutation Test (200 vòng).
2. **Đánh giá phân khúc (Subgroup Audit):** Hiệu năng theo từng sân bay (`SGN, HAN, DAD, CXR, PQC`), từng hãng bay, khung giờ cao điểm và điều kiện thời tiết (kèm cảnh báo cỡ mẫu nhỏ `n < 30`).
3. **SHAP Explainability & Model Governance:** Kiểm tra `model_registry.json`, trạng thái `Production`, `checksums.sha256` và tính toàn vẹn `verify_bundle_checksums`.

In [1]:
import json
from pathlib import Path
import pandas as pd

from flight_delay.common.constants import ARTIFACT_MANIFESTS_DIR, ROOT_DIR
from flight_delay.governance.artifact_manager import verify_bundle_checksums

ev = json.loads(Path("reports/experiments/evaluation_report.json").read_text(encoding="utf-8"))
reg_man = json.loads((ARTIFACT_MANIFESTS_DIR / "model_registry.json").read_text(encoding="utf-8"))
cls = ev["classification_3class"]

print("=== Statistical Comparisons & Permutation Test ===")
print("Permutation Test:", json.dumps(cls["permutation_test"], indent=2))
print("Bootstrap vs Heuristic:", json.dumps(cls["statistical_comparisons"]["best_vs_heuristic"], indent=2))

print("\n=== Subgroup Performance by Airport (Holdout Test) ===")
print(pd.DataFrame(cls["subgroups"]["by_airport"]).T[["count", "accuracy", "balanced_accuracy", "macro_f1", "mae_model", "small_sample_warning"]].to_string())

print("\n=== Subgroup Performance by Airline (Holdout Test) ===")
print(pd.DataFrame(cls["subgroups"]["by_airline"]).T[["count", "accuracy", "balanced_accuracy", "macro_f1", "small_sample_warning"]].to_string())

print("\n=== Governance Production Registry & SHA-256 Verification ===")
for name, entry in reg_man["production"].items():
    bdir = ROOT_DIR / entry["bundle_dir"]
    ok_csum = verify_bundle_checksums(bdir)
    print(f"  - {name:34s} | ver={entry['version']} | alg={entry['algorithm']:25s} | SHA256_OK={ok_csum}")

=== Statistical Comparisons & Permutation Test ===
Permutation Test: {
  "n_permutations": 200,
  "metric": "balanced_accuracy",
  "model_score": 41.86,
  "chance_level_mean": 33.35,
  "chance_level_std": 0.61,
  "p_value": 0.0,
  "statistically_significant": true
}
Bootstrap vs Heuristic: {
  "bootstrap_f1_macro": {
    "metric": "f1_macro",
    "n_bootstrap": 400,
    "mean_diff_b_minus_a": 0.0455,
    "ci_95": [
      0.0292,
      0.0734
    ],
    "significant_005": true
  },
  "mcnemar": {
    "n01_b_better": 257,
    "n10_a_better": 222,
    "statistic": 2.4134,
    "p_value": 0.120304,
    "significant_005": false
  }
}

=== Subgroup Performance by Airport (Holdout Test) ===
    count accuracy balanced_accuracy macro_f1 mae_model small_sample_warning
SGN  1206     74.3             64.77    64.39     16.48                False
HAN   933    75.24             62.71     60.5     13.25                False
DAD   267    72.28             38.19    38.23     12.48                False
